# Olympiad Task: Double Agent Faces

Adapted from **Double Agent Dilemma**, IOAI 2026 Individual Contest, Day 2 (CC BY 4.0). The website has the full brief and hints.

Put this notebook in the same folder as `emoji_faces.py`, `agent_m.pth` and `agent_c.pth`. Click a cell and press **Shift + Enter** to run it. Work from top to bottom.

- **PREDICT**: type your guess as a comment before you run the cell
- **CHANGE ONE THING**: change the value it names, run it again, and compare
- **FINISH THE LINES**: replace each `______`
- **WRITE IT YOURSELF**: write the code under each comment
- **CHECK**: tries your function on 10 faces and tells you what it finds

## 1. Set Up
This loads the two agents and picks 100 practice faces and 100 test faces. Both agents get every one of these faces right.

In [ ]:
import math
import time
import torch
from torch import nn
import torch.nn.functional as F
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from emoji_faces import make_face, CLASSES

to_tensor = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])   # picture to numbers from 0 to 1

def prepare(picture):
    # the Normalize step from get_ready: numbers from 0 to 1 become -1 to 1
    return (picture - 0.5) / 0.5

class MoodNet(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(3 * 32 * 32, hidden)
        self.hidden2 = nn.Linear(hidden, 128)
        self.output = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        return self.output(x)

class MoodCNN(nn.Module):
    def __init__(self, channels=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, channels, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(channels, 2 * channels, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classify = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2 * channels * 8 * 8, 64), nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.classify(self.features(x))

agent_m = MoodNet()
agent_m.load_state_dict(torch.load("agent_m.pth", weights_only=True))
agent_c = MoodCNN()
agent_c.load_state_dict(torch.load("agent_c.pth", weights_only=True))
for agent in [agent_m, agent_c]:
    agent.eval()
    for p in agent.parameters():
        p.requires_grad = False          # the agents never learn in this task

def guess(agent, picture):
    with torch.no_grad():
        return agent(prepare(picture).unsqueeze(0)).argmax(dim=1).item()

def pick_faces(first_seed, how_many=100):
    # draw faces from first_seed onwards, and keep only the ones BOTH agents get right
    faces = []
    seed = first_seed
    while len(faces) < how_many:
        picture, mood = make_face(seed)
        picture = to_tensor(picture)
        if guess(agent_m, picture) == mood and guess(agent_c, picture) == mood:
            faces.append((picture, mood))
        seed += 1
    return faces

practice_faces = pick_faces(2_000_000)
test_faces = pick_faces(2_100_000)
print("Both agents loaded.", len(practice_faces), "practice faces and", len(test_faces), "test faces.")

## 2. The Tools
- `works(picture, mood, keep, fool, change)`: True if `keep` is still right and `fool` is wrong
- `size(change)`: how big the change is, in brightness steps out of 255
- `score(changes_a, changes_b, faces)`: the official score
- `run_all(make_change, faces)`: runs your method on every face, both types
- `show(picture, mood, change)`: draws a face before and after
- `check(make_change)`: tries your method on 10 faces and gives feedback

In [ ]:
def works(picture, mood, keep, fool, change):
    changed = (picture + change).clamp(0, 1)      # add the change, keep every value between 0 and 1
    return guess(keep, changed) == mood and guess(fool, changed) != mood

def size(change):
    # the typical change to one colour value, in brightness steps out of 255
    return change.pow(2).mean().sqrt().item() * 255

def marks(changes_a, changes_b, faces):
    a = 0
    b = 0
    sizes = []
    for (picture, mood), change_a, change_b in zip(faces, changes_a, changes_b):
        if works(picture, mood, agent_m, agent_c, change_a):    # Type A: keep M right, fool C
            a += 1
        if works(picture, mood, agent_c, agent_m, change_b):    # Type B: keep C right, fool M
            b += 1
        sizes.append(size(change_a))
        sizes.append(size(change_b))
    pure = (a + b) / (2 * len(faces))
    average_size = sum(sizes) / len(sizes)
    penalty = 1.5 - 1 / (1 + math.exp(-0.2 * average_size))     # 1.0 for no change, falling towards 0.5
    return pure * penalty, a, b, average_size, penalty

def score(changes_a, changes_b, faces):
    final, a, b, average_size, penalty = marks(changes_a, changes_b, faces)
    print("Type A worked:", a, "of", len(faces), "   Type B worked:", b, "of", len(faces))
    print("Average size:", round(average_size, 2), "steps   Penalty factor:", round(penalty, 3))
    print("FINAL SCORE:", round(final, 3))

def run_all(make_change, faces):
    start = time.time()
    changes_a = []
    changes_b = []
    for picture, mood in faces:
        changes_a.append(make_change(picture, mood, agent_m, agent_c))   # Type A
        changes_b.append(make_change(picture, mood, agent_c, agent_m))   # Type B
    seconds = time.time() - start
    print("Took", round(seconds, 1), "seconds for", len(faces), "faces")
    return changes_a, changes_b, seconds

def show(picture, mood, change):
    changed = (picture + change).clamp(0, 1)
    plt.figure(figsize=(8, 3))
    plt.subplot(1, 3, 1); plt.imshow(picture.permute(1, 2, 0)); plt.axis("off")
    plt.title("original\nM: " + CLASSES[guess(agent_m, picture)] + "  C: " + CLASSES[guess(agent_c, picture)], fontsize=9)
    plt.subplot(1, 3, 2); plt.imshow((0.5 + 20 * change).clamp(0, 1).permute(1, 2, 0)); plt.axis("off")
    plt.title("the change x 20\nsize " + str(round(size(change), 1)) + " steps", fontsize=9)
    plt.subplot(1, 3, 3); plt.imshow(changed.permute(1, 2, 0)); plt.axis("off")
    plt.title("changed\nM: " + CLASSES[guess(agent_m, changed)] + "  C: " + CLASSES[guess(agent_c, changed)], fontsize=9)
    plt.show()

def check(make_change, ours=""):
    # tries your function on the first 10 practice faces and says what it finds
    faces = practice_faces[:10]
    a = 0
    b = 0
    zeros = 0
    sizes = []
    start = time.time()
    for picture, mood in faces:
        for keep, fool in [(agent_m, agent_c), (agent_c, agent_m)]:
            try:
                change = make_change(picture, mood, keep, fool)
            except Exception as problem:
                print("Your function stopped with an error:", type(problem).__name__, problem)
                print("Is there still a ______ to fill in, or a cell above that you haven't run?")
                return
            if not torch.is_tensor(change) or tuple(change.shape) != (3, 32, 32):
                print("Your function must return a tensor with shape (3, 32, 32).")
                return
            if change.abs().max().item() == 0:
                zeros += 1
            if works(picture, mood, keep, fool, change):
                if keep is agent_m:
                    a += 1
                else:
                    b += 1
            sizes.append(size(change))
    per_100 = (time.time() - start) * 10
    print("On 10 faces: Type A worked", a, "times, Type B", b, "times. Average size", round(sum(sizes) / len(sizes), 2))
    if ours:
        print("Ours on the same 10 faces:", ours)
    if zeros == 20:
        print("Every change is zero. Is your function still the starter version, ending in return torch.zeros_like(picture)?")
    elif a + b == 0:
        print("Nothing worked yet. Try your idea on one face with works() and show() first.")
    elif per_100 > 300:
        print("Too slow: about", round(per_100), "seconds for 100 faces. The limit is 300.")
    else:
        print("CHECK passed. About", round(per_100), "seconds for 100 faces. Now run it on all the practice faces below.")

## 3. Warm-Up: One Face by Hand
Before writing any method, try changes on one face and watch what the agents do. `keep` is the agent that must stay right, and `fool` is the one that must go wrong. For Type A, `keep` is `agent_m` and `fool` is `agent_c`.

In [ ]:
# PREDICT: this adds random speckles to one face. Will either agent change its mind?
# My guess:
torch.manual_seed(0)
picture, mood = practice_faces[0]
change = 0.02 * torch.randn_like(picture)        # random speckles, about 5 steps out of 255
print("Right mood:", CLASSES[mood], "   size:", round(size(change), 1), "steps")
print("Type A works?", works(picture, mood, agent_m, agent_c, change))
print("Type B works?", works(picture, mood, agent_c, agent_m, change))
show(picture, mood, change)

In [ ]:
# CHANGE ONE THING: make the speckles much stronger, 0.02 -> 0.3. What happens to the agents now?
change = 0.3 * torch.randn_like(picture)
print("size:", round(size(change), 1), "steps")
print("Type A works?", works(picture, mood, agent_m, agent_c, change))
print("Type B works?", works(picture, mood, agent_c, agent_m, change))
show(picture, mood, change)

Small random speckles fool nobody. Big ones might split the agents, but at that size anyone can see them. The task is to fool **exactly one** agent with a change nobody can see. That needs a method that chooses which pixels to change.

## Experiment Log

Fill in a row every time you score something. Double-click this cell to edit it.

| Level or idea | What I changed | Practice score | Test score | Seconds |
| --- | --- | --- | --- | --- |
| Baseline | nothing | 0.000 | | |
| | | | | |
| | | | | |
| | | | | |

## 4. The Baseline
The baseline adds the same tiny amount to every pixel. It runs and it's small, but it fools nobody: it scores 0.

In [ ]:
def baseline(picture, mood, keep, fool):
    return torch.full_like(picture, 0.0001)      # the same tiny change everywhere

changes_a, changes_b, seconds = run_all(baseline, practice_faces)
score(changes_a, changes_b, practice_faces)

## 5. Level 1: Cover a Square
No gradients yet. Paint a small square of the face in its own average colour, and see if that splits the agents. First, one square by hand.

In [ ]:
# CHANGE ONE THING: move the square (top, left) or change its side. Can you make Type A or Type B work?
# Hint: squares over the mouth, around top = 20, are a good place to start.
top = 14
left = 8
side = 8
change = torch.zeros_like(picture)
patch = picture[:, top:top + side, left:left + side]
flat = patch.mean(dim=(1, 2), keepdim=True)             # the square's average colour
change[:, top:top + side, left:left + side] = flat - patch
print("Type A works?", works(picture, mood, agent_m, agent_c, change))
print("Type B works?", works(picture, mood, agent_c, agent_m, change))
show(picture, mood, change)

Trying positions by hand is slow. Now write a function that tries every position and size for you, and keeps the smallest change that works.

In [ ]:
# WRITE IT YOURSELF. Comment guide:
def cover_square(picture, mood, keep, fool):
    # 1. Start with no change: best = torch.zeros_like(picture). Send this if nothing works.
    # 2. Try square sizes, smallest first:  for side in [4, 6, 8, 10, 12]:
    # 3. Try every position, two pixels apart:  for top in range(0, 33 - side, 2):  (and the same for left)
    # 4. Make a change that paints the square in its own average colour:
    #      patch = picture[:, top:top + side, left:left + side]
    #      the change is (average colour - patch) inside the square, and 0 everywhere else
    # 5. If works(picture, mood, keep, fool, change), keep it if it's the smallest so far.
    # 6. After each square size, if something worked, return it.
    return torch.zeros_like(picture)

In [ ]:
# CHECK
check(cover_square, ours="Type A 8, Type B 10, average size 9.09")

In [ ]:
changes_a, changes_b, seconds = run_all(cover_square, practice_faces)
score(changes_a, changes_b, practice_faces)

## 6. Level 2: Let the Gradients Choose
This is the new idea. The agents' weights stay fixed, and the **change** is the value that learns. The loss is small when `keep` is sure of the right mood and `fool` is far from it, so going downhill pushes the agents apart.

First, watch it work on one face, step by step.

In [ ]:
# PREDICT: how many steps to fool Agent C on this face while Agent M stays right (Type A)?
# My guess:
picture, mood = practice_faces[3]
change = torch.zeros_like(picture, requires_grad=True)    # the change learns
optimiser = torch.optim.Adam([change], lr=0.01)
right = torch.tensor([mood])
for step in range(50):
    changed = prepare((picture + change).clamp(0, 1)).unsqueeze(0)
    loss = F.cross_entropy(agent_m(changed), right) - F.cross_entropy(agent_c(changed), right)
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    now = (picture + change.detach()).clamp(0, 1)
    print("step", step + 1, "  M says", CLASSES[guess(agent_m, now)], "  C says", CLASSES[guess(agent_c, now)],
          "  size", round(size(change.detach()), 2))
    if works(picture, mood, agent_m, agent_c, change.detach()):
        print("It works!")
        break
show(picture, mood, change.detach())

Now turn that into a function that works for either type. `keep` and `fool` take the places of `agent_c` and `agent_m`.

In [ ]:
# FINISH THE LINES. Replace each ______.
def nudge(picture, mood, keep, fool, steps=100, lr=0.01):
    change = torch.zeros_like(picture, requires_grad=______)   # the change is the value that learns
    optimiser = torch.optim.Adam([change], lr=lr)              # Adam updates the change, not the agents
    right = torch.tensor([mood])
    for step in range(steps):
        if works(picture, mood, keep, fool, change.detach()):  # stop the moment it works
            break
        changed = prepare((picture + change).clamp(0, 1)).unsqueeze(0)
        # keep's cross-entropy MINUS fool's cross-entropy, as in the cell above
        loss = ______
        optimiser.zero_grad()
        loss.______()
        optimiser.______()
    return change.detach()

In [ ]:
# CHECK
check(nudge, ours="Type A 8, Type B 10, average size 8.33")

In [ ]:
changes_a, changes_b, seconds = run_all(nudge, practice_faces)
score(changes_a, changes_b, practice_faces)

In [ ]:
# CHANGE ONE THING: try nudge with a smaller learning rate and more steps.
def nudge_slow(picture, mood, keep, fool):
    return nudge(picture, mood, keep, fool, steps=200, lr=0.002)

changes_a, changes_b, seconds = run_all(nudge_slow, practice_faces)
score(changes_a, changes_b, practice_faces)

## 7. Level 3: Make It Smaller
Once a change works, try a fraction of it. Halve the gap each time to find the smallest fraction that still works. Also, send no change at all when it didn't work: a failed change still counts towards the size.

In [ ]:
# PREDICT: this Type B change works. Does half of it still work? A quarter? A tenth?
# My guess:
picture, mood = practice_faces[0]
change = nudge(picture, mood, agent_c, agent_m)
for fraction in [1.0, 0.5, 0.25, 0.1]:
    print("fraction", fraction, "  size", round(size(change * fraction), 2),
          "  works?", works(picture, mood, agent_c, agent_m, change * fraction))

In [ ]:
# FINISH THE LINES, then WRITE IT YOURSELF.
def make_smaller(picture, mood, keep, fool, change):
    # Find the smallest fraction of the change that still works, by halving the gap each time.
    low = 0.0                   # no change: doesn't work
    high = 1.0                  # the whole change: works
    for attempt in range(12):
        middle = (low + high) / 2
        if works(picture, mood, keep, fool, change * middle):
            high = ______       # still works: try smaller
        else:
            low = ______        # stopped working: go back up
    return change * high

def nudge_then_shrink(picture, mood, keep, fool):
    # 1. change = nudge(...) with a smaller learning rate and more steps: steps=200, lr=0.002
    # 2. If the change doesn't work, return torch.zeros_like(picture). A failed change still adds to the size.
    # 3. Otherwise return make_smaller(...) of it.
    return torch.zeros_like(picture)

In [ ]:
# CHECK
check(nudge_then_shrink, ours="Type A 9, Type B 10, average size 5.04")

In [ ]:
changes_a, changes_b, seconds = run_all(nudge_then_shrink, practice_faces)
score(changes_a, changes_b, practice_faces)

## 8. Level 4: Keep Squeezing
Don't stop when it works. Each step, if the change works, remember it and make it smaller. If it stops working, nudge again. Keep the smallest change that worked.

In [ ]:
# FINISH THE LINES. The outline is done: fill in the two losses and the ending.
def nudge_and_squeeze(picture, mood, keep, fool, steps=300, lr=0.002):
    change = torch.zeros_like(picture, requires_grad=True)
    optimiser = torch.optim.Adam([change], lr=lr)
    right = torch.tensor([mood])
    best = None
    for step in range(steps):
        changed = prepare((picture + change).clamp(0, 1)).unsqueeze(0)
        if works(picture, mood, keep, fool, change.detach()):
            if best is None or size(change.detach()) < size(best):
                best = change.detach().clone()           # remember the smallest change that works
            loss = ______                                # it works: a loss that is smallest when change is all 0
        else:
            loss = ______                                # it doesn't work: the same loss as nudge
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    # If best is None, nothing worked: return torch.zeros_like(picture)
    # Otherwise squeeze the last bit out: return make_smaller(picture, mood, keep, fool, best)
    return torch.zeros_like(picture)

In [ ]:
# CHECK
check(nudge_and_squeeze, ours="Type A 9, Type B 10, average size 2.13")

In [ ]:
changes_a, changes_b, seconds = run_all(nudge_and_squeeze, practice_faces)
score(changes_a, changes_b, practice_faces)
picture, mood = practice_faces[0]
show(picture, mood, changes_a[0])

## 9. Your Own Ideas
Type A (keep M, fool C) is harder than Type B. Look at the faces where it fails. Ideas to try, one at a time:

- more steps, or a different learning rate, for Type A only (`keep is agent_m` is True for Type A)
- push the `fool` agent towards one particular wrong mood
- a weight in the loss: `2 * F.cross_entropy(keep(...))` or `2 * F.cross_entropy(fool(...))`
- change only some colours, or only the middle of the face

Record every attempt in the Experiment Log.

In [ ]:
# Which faces does Type A fail on? Run this, then look at a few with show().
failed = []
for i, (picture, mood) in enumerate(practice_faces):
    if not works(picture, mood, agent_m, agent_c, changes_a[i]):
        failed.append(i)
print("Type A failed on faces", failed)
print("Their moods:", [CLASSES[practice_faces[i][1]] for i in failed])

In [ ]:
# YOUR TURN: write your own make_change function here. Run check() on it, then score it on the practice faces.

## 10. Check on the Test Faces
When you have chosen your best method, score it **once** on the test faces. These are faces you haven't tuned on, so this is a fairer guess of your contest score.

In [ ]:
# Put the name of your best function in place of nudge_and_squeeze, then run once.
changes_a, changes_b, seconds = run_all(nudge_and_squeeze, test_faces)
score(changes_a, changes_b, test_faces)

## 11. Contest
Your teacher gives out a `FIRST_SEED` when the contest starts. That draws 100 new faces nobody has seen. Your method has **5 minutes** to make all its changes. Hand in the file this cell saves.

In [ ]:
MY_NAME = "your-name"
FIRST_SEED = 0               # your teacher gives you this number when the contest starts
my_best = nudge_and_squeeze  # the function you want marked: no brackets

if FIRST_SEED == 0:
    print("Ask your teacher for the contest FIRST_SEED, put it in above, then run this cell again.")
else:
    contest_faces = pick_faces(FIRST_SEED)
    changes_a, changes_b, seconds = run_all(my_best, contest_faces)
    if seconds > 300:
        print("Over the 5 minute limit. Make your method faster and run again.")
    else:
        submission = {
            "name": MY_NAME,
            "first_seed": FIRST_SEED,
            "seconds": seconds,
            "changes_a": torch.stack(changes_a),
            "changes_b": torch.stack(changes_b),
        }
        torch.save(submission, "submission-" + MY_NAME + ".pt")
        print("Saved submission-" + MY_NAME + ".pt. Hand this file in.")